Soma total de vendas por dia, ordens com status de DELIVERED e state = NY

In [0]:
bronze_path = '/Volumes/bikestore/bronze/bronze_volumes'
silver_path = '/Volumes/bikestore/silver/silver_volumes'
gold_path = '/Volumes/bikestore/gold/gold_volumes'

In [0]:
silver_tables = {
    'tmp_silver_customers' : f'{silver_path}/customer',
    'tmp_silver_orders' : f'{silver_path}/orders',
    'tmp_silver_products' : f'{silver_path}/product',
}

for view_name, path in silver_tables.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name))

In [0]:
df_sales_ny_gold = spark.sql("""
SELECT
    shipped_date,
    round(sum(total_sale), 2) as total_sale
FROM tmp_silver_orders
WHERE state = 'NY'
AND status = 'Delivered'
AND shipped_date IS NOT NULL
GROUP BY shipped_date
""")

df_sales_ny_gold.write.mode("overwrite").format("delta").option(
    "mergeSchema", "true"
).save(gold_path + "/sales_ny")

In [0]:
%sql
-- Criando uma tabela física

CREATE TABLE IF NOT EXISTS bikestore.gold.sales_ny
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/bikestore/gold/gold_volumes/sales_ny`;